# Stock Trend Pattern Recognition — MLOps with Amazon SageMaker

This project rebuilds the stock-trend-lstm model using Amazon SageMaker as the ML platform. The model architecture is intentionally identical — the same stacked LSTM trained on the same data. The difference is entirely in the infrastructure: training runs as a managed SageMaker Job, the model is versioned in SageMaker Model Registry, and inference is served via a managed Real-Time Endpoint. This is the difference between a data scientist running a notebook and an MLOps engineer deploying a production ML system.

> **Pattern detection only — not financial advice. Past patterns do not guarantee future results.**

**Tickers**: AAPL · MSFT · GOOGL  
**Sequence length**: 60 trading days  
**Prediction horizon**: 5 trading days  
**Model**: Stacked LSTM — identical architecture to stock-trend-lstm  
**Platform**: Amazon SageMaker (Training Job + Model Registry + Real-Time Endpoint)  
**Frontend**: Flask on Azure App Service → calls SageMaker endpoint via boto3

In [ ]:
import json
from pathlib import Path
from IPython.display import display, Image
import pandas as pd

from config import (
    S3_BUCKET, S3_PREFIX, REGION, ROLE_NAME,
    TICKERS, SEQUENCE_LENGTH, PREDICTION_HORIZON, TRAIN_SPLIT,
    TRAINING_INSTANCE, ENDPOINT_INSTANCE, ENDPOINT_NAME,
    MODEL_PACKAGE_GROUP, MODEL_APPROVAL_STATUS,
    MODEL_DIR, PLOTS_DIR, DATA_DIR,
    BEDROCK_MODEL_ID, BEDROCK_REGION,
)

print('Config loaded')
print(f'  Tickers:           {TICKERS}')
print(f'  Sequence length:   {SEQUENCE_LENGTH} days')
print(f'  Prediction horizon:{PREDICTION_HORIZON} days')
print(f'  Training instance: {TRAINING_INSTANCE}')
print(f'  Endpoint instance: {ENDPOINT_INSTANCE}')
print(f'  S3 bucket:         {S3_BUCKET}')
print(f'  Region:            {REGION}')

## MLOps vs Ad-Hoc ML

In stock-trend-lstm, training ran locally in Jupyter. The model was saved as a `.keras` file and committed to GitHub. Inference ran in a Flask app on the same machine as the code. This works for a portfolio project — it does not work for a production system serving thousands of requests.

SageMaker addresses three production requirements that the local approach cannot meet:

1. **Reproducibility**: every training run is a tracked job with logged hyperparameters, metrics, and artefact location. You can reproduce any previous model exactly.
2. **Scalability**: the endpoint auto-scales based on traffic. A Flask app on a single VM cannot do this.
3. **Separation of concerns**: training, registration, and serving are separate steps with separate infrastructure. A data scientist can retrain without touching the serving infrastructure.

## AWS Architecture

```
[Local: prepare_data.py]
       |
       v
[S3: training data]
       |
       v
[SageMaker Training Job] <── [run_training_job.py]
       |
       v
[S3: model artefacts]
       |
       v
[SageMaker Model Registry] <── [register_model.py]
       |
       v
[SageMaker Real-Time Endpoint] <── [deploy_endpoint.py]
       |
       v
[Flask on Azure App Service] ──> calls endpoint via boto3
       |
       v
[User browser]
```

Each step writes metadata consumed by the next:
- `prepare_data.py` → S3 URIs for train/test data
- `run_training_job.py` → `models/training_job_metadata.json` (job name, model artefact S3 URI)
- `register_model.py` → `models/model_package_arn.txt`
- `deploy_endpoint.py` → `models/endpoint_metadata.json` (endpoint name, URL)

In [ ]:
# Step 1: Prepare data and upload to S3
# This downloads 5 years of data, engineers features, and uploads to S3.
# Runtime: ~2 minutes (network-bound)

%run prepare_data.py

In [ ]:
# Display EDA plots
for plot in ['01_price_history.png', '02_technical_indicators.png',
             '03_label_distribution.png', '04_feature_correlation.png']:
    path = PLOTS_DIR / plot
    if path.exists():
        print(f'\n{plot}')
        display(Image(filename=str(path)))

## SageMaker Training Jobs

A SageMaker Training Job spins up a managed EC2 instance, installs the specified framework (TensorFlow 2.15), copies the training data from S3, runs `train_sagemaker.py`, saves the model artefact back to S3, and terminates the instance. You pay only for the time the training instance runs — typically 5–15 minutes for this dataset on ml.m5.xlarge.

**Key conventions** in `training/train_sagemaker.py`:
- Data paths come from **environment variables** (`SM_CHANNEL_TRAIN`, `SM_CHANNEL_TEST`), not hardcoded paths
- Hyperparameters arrive via **argparse**, not constants
- The model must be saved in **TensorFlow SavedModel format** at `SM_MODEL_DIR/1/` (the `1` is the TF Serving version number)
- `stdout` is captured as **CloudWatch training logs** — print statements are your logging
- There is no `if __name__ == '__main__'` guard — SageMaker calls the script directly

**Cost**: ml.m5.xlarge is ~$0.23/hour. A 15-minute job costs ~$0.06.

In [ ]:
# Step 2: Launch SageMaker Training Job
# wait=True streams live CloudWatch logs below.
# Runtime: ~15 minutes on ml.m5.xlarge
# If job fails, check: AWS Console → SageMaker → Training → Training jobs

%run run_training_job.py

In [ ]:
# Inspect training job metadata
with open(MODEL_DIR / 'training_job_metadata.json') as f:
    tj = json.load(f)
print(json.dumps(tj, indent=2))

## SageMaker Model Registry

The Model Registry solves a problem that every production ML team faces: which version of the model is currently deployed, and can we roll back?

Each training run creates a versioned **Model Package** with its artefact location, inference specification, and approval status. Deploying a new model means creating a new package version and updating the endpoint — the old version remains in the registry and can be redeployed in minutes.

This is the production version of the `model_registry.csv` pattern used in telco-churn-predictor. The principle is identical; the implementation is managed rather than manual.

The `InferenceSpecification` in `register_model.py` tells SageMaker exactly how to serve the model: which Docker image (TensorFlow Serving 2.15), where the artefact lives in S3, what content type it accepts, and which instance types are supported.

In [ ]:
# Step 3: Register model in Model Registry
%run register_model.py

In [ ]:
# Verify ARN was saved
arn_path = MODEL_DIR / 'model_package_arn.txt'
if arn_path.exists():
    print(f'Model package ARN: {arn_path.read_text().strip()}')

## SageMaker Real-Time Endpoints

A Real-Time Endpoint is a persistent HTTPS endpoint backed by a managed EC2 instance running TensorFlow Serving. It accepts JSON payloads and returns predictions with ~100ms latency. Unlike AWS Lambda, there is no cold start — the model is loaded once at endpoint creation and stays in memory. Unlike a Flask app on App Service, the endpoint auto-scales and is monitored by SageMaker natively.

The endpoint URL follows the pattern:
```
https://runtime.sagemaker.{region}.amazonaws.com/endpoints/{endpoint-name}/invocations
```

> **IMPORTANT**: real-time endpoints bill by the hour. `ml.t2.medium` costs ~$0.065/hour. Delete the endpoint after the portfolio demo using `delete_endpoint.py`.

In [ ]:
# Step 4: Deploy Real-Time Endpoint
# Runtime: ~10 minutes
# REMEMBER: run delete_endpoint.py after recording the demo URL

%run deploy_endpoint.py

In [ ]:
# Inspect endpoint metadata
with open(MODEL_DIR / 'endpoint_metadata.json') as f:
    ep = json.load(f)
print(json.dumps(ep, indent=2))

## Multi-Cloud Architecture

The Flask frontend runs on Azure App Service and calls the SageMaker endpoint via boto3 over HTTPS. This is a deliberate architectural choice that mirrors enterprise multi-cloud deployments. Accenture's enterprise clients frequently run application infrastructure on Azure (Microsoft's platform, which Accenture has a deep partnership with) while running ML workloads on AWS SageMaker (which has the most mature managed ML platform).

The two clouds communicate via HTTPS — no VPN or peering required for this traffic pattern. AWS credentials (`AWS_ACCESS_KEY_ID`, `AWS_SECRET_ACCESS_KEY`, `AWS_DEFAULT_REGION`) are stored as Azure App Service Application Settings — never in code or committed to Git. This is the standard pattern for cross-cloud authentication.

The Flask app (`app.py`) does no local inference — every prediction call goes to the SageMaker endpoint via boto3's `sagemaker-runtime` client. The app only handles request parsing, response formatting, and the MA crossover baseline signal.

## EDA

In [ ]:
for plot in ['01_price_history.png', '02_technical_indicators.png',
             '03_label_distribution.png', '04_feature_correlation.png']:
    path = PLOTS_DIR / plot
    if path.exists():
        print(f'\n{plot}')
        display(Image(filename=str(path)))

## Training Results and Evaluation

The metrics below are captured from `models/model_metrics.json`, written by the SageMaker training container. Unlike a local notebook where metrics are computed inline, SageMaker metrics are captured from CloudWatch logs and output artefacts — they exist independently of the training environment.

In [ ]:
metrics_path = MODEL_DIR / 'model_metrics.json'
if metrics_path.exists():
    with open(metrics_path) as f:
        metrics = json.load(f)

    rows = [(k, v) for k, v in metrics.items() if isinstance(v, float)]
    df_metrics = pd.DataFrame(rows, columns=['Metric', 'Value'])
    df_metrics['Value'] = df_metrics['Value'].round(4)
    display(df_metrics)
else:
    print('models/model_metrics.json not found — run the training job first.')

In [ ]:
# Evaluation plots
for plot in ['05_training_curves.png', '06_roc_curve.png',
             '07_model_comparison.png', '08_predictions_vs_actual.png']:
    path = PLOTS_DIR / plot
    if path.exists():
        print(f'\n{plot}')
        display(Image(filename=str(path)))

## Amazon Bedrock Comparator

This is the fourth project benchmarking Bedrock against trained models. In this project, Bedrock is compared against a SageMaker-hosted LSTM rather than a locally-trained model — the inference pipeline is different but the comparison methodology is identical.

Bedrock (Claude Haiku) is given a zero-shot prompt containing the last 5 days of key indicators and asked to predict the 5-day trend direction. The LSTM uses the full 60-day sequence. The comparison is deliberately asymmetric — it illustrates the trade-off between infrastructure complexity (LSTM) and deployment simplicity (Bedrock API call).

**Across four projects:**

| Project | Task | Finding |
|---|---|---|
| telco-churn-predictor | Tabular classification | XGBoost outperforms Bedrock |
| spam-classifier | Short text | Results similar |
| imdb-sentiment-classifier | Long text | LSTM competitive with Bedrock |
| stock-trend-sagemaker | Time series | TBD |

In [ ]:
# Run Bedrock comparator — requires live AWS credentials and Bedrock access
%run bedrock_comparator.py

In [ ]:
# Display bedrock comparison results
results_path = MODEL_DIR / 'bedrock_results.json'
if results_path.exists():
    with open(results_path) as f:
        br = json.load(f)

    comparison = pd.DataFrame([
        {'Model': 'SageMaker LSTM', 'Accuracy': br.get('lstm_accuracy', 'TBD'),
         'F1': br.get('lstm_f1', 'TBD'), 'ROC-AUC': br.get('lstm_roc_auc', 'TBD')},
        {'Model': 'Bedrock Claude Haiku', 'Accuracy': br.get('bedrock_accuracy', 'TBD'),
         'F1': br.get('bedrock_f1', 'TBD'), 'ROC-AUC': br.get('bedrock_roc_auc', 'TBD')},
    ])
    display(comparison)

## MLOps Pipeline Comparison — SageMaker vs Manual

| Concern | telco-churn-predictor | stock-trend-sagemaker |
|---|---|---|
| Training | Local Jupyter notebook | SageMaker Training Job |
| Model versioning | model_registry.csv | SageMaker Model Registry |
| Artefact storage | models/ in Git repo | S3 bucket |
| Serving | Flask on App Service | SageMaker Real-Time Endpoint |
| Scaling | Manual (VM size) | Auto-scaling (managed) |
| Monitoring | None | CloudWatch + SageMaker |
| Rollback | Re-deploy from Git | Endpoint update from Registry |
| Cost model | Always-on VM | Pay-per-training + per-hour |

The manual pattern (telco-churn-predictor) is appropriate for a portfolio project or a small internal tool. The SageMaker pattern is appropriate for a production system. Understanding when to use each — and being able to implement both — is what the MLOps Engineer title requires.

## Key Findings

*TBD — fill after running the full pipeline.*

- LSTM ROC-AUC: TBD
- Training job duration: TBD
- Endpoint smoke test latency: TBD
- Bedrock vs LSTM accuracy: TBD

## Cost Summary

Approximate AWS cost for this project:

| Item | Cost |
|---|---|
| SageMaker Training Job (ml.m5.xlarge, ~15 min) | ~$0.30 |
| S3 storage (model artefacts + data, <500 MB) | <$0.01 |
| SageMaker Endpoint (ml.t2.medium, per hour while running) | ~$0.065/hr |
| Bedrock API calls (~30 calls, Haiku pricing) | ~$0.01 |
| **Total for development + portfolio demo** | **<$2.00** |

Delete the endpoint after recording the demo URL to stop charges:
```bash
python delete_endpoint.py
```

## Azure App Service Deployment

The Flask frontend on Azure App Service calls the SageMaker endpoint via boto3. AWS credentials (`AWS_ACCESS_KEY_ID`, `AWS_SECRET_ACCESS_KEY`, `AWS_DEFAULT_REGION`) are stored as App Service Application Settings — never in code or committed to Git. This is the standard pattern for cross-cloud authentication.

```bash
az group create \
  --name stock-trend-sm-rg \
  --location westeurope

az appservice plan create \
  --name stock-trend-sm-plan \
  --resource-group stock-trend-sm-rg \
  --sku B1 --is-linux
# Scale to F1 via portal after creation

az webapp create \
  --name stock-trend-sagemaker \
  --resource-group stock-trend-sm-rg \
  --plan stock-trend-sm-plan \
  --runtime "PYTHON:3.11"

az webapp config set \
  --name stock-trend-sagemaker \
  --resource-group stock-trend-sm-rg \
  --startup-file "gunicorn --bind=0.0.0.0:8000 --timeout 600 app:app"

az webapp config appsettings set \
  --name stock-trend-sagemaker \
  --resource-group stock-trend-sm-rg \
  --settings \
    SCM_DO_BUILD_DURING_DEPLOYMENT=true \
    AWS_ACCESS_KEY_ID=<your-key> \
    AWS_SECRET_ACCESS_KEY=<your-secret> \
    AWS_DEFAULT_REGION=us-east-1

cd stock-trend-sagemaker && zip -r deploy.zip . \
  -x "*.git*" -x "venv/*" -x "__pycache__/*" -x "*.ipynb_checkpoints*"

az webapp deployment source config-zip \
  --name stock-trend-sagemaker \
  --resource-group stock-trend-sm-rg \
  --src deploy.zip
```